# 01b — EffNet-B0 Block-Shuffled Null Distribution (20 Seeds)

**Testing the Split Ordering Effect (GATE 1 Follow-up)**

Hypothesis Test:
- Single `block_contiguous` accuracy for `effnet_b0`: 81.23% (5 model-init seeds on identical split).
- Null Distribution: 20 seeds of `effnet_b0` on `block_shuffled` (both model-init and split variance).
- Decision Rule: Locate 81.23% in the empirical CDF of the 20 `block_shuffled` seeds.
  - If 81.23% falls inside the central 95% [2.5%, 97.5%]: NO position/ordering effect (one unlucky draw; Option 2 dead).
  - If 81.23% falls outside: Ordering effect is real (reframe as split ordering choice).
  - Borderline halt condition: Percentile in [90.0%, 97.5%].


In [ ]:
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "GPU accelerator required!"


In [ ]:
SEEDS = [42, 123, 2024, 7, 2718, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16]
MODEL = "effnet_b0"
STRAT = "block_shuffled"
CONTIGUOUS_MEAN_ACC = 81.23

OUT = f"{RESULTS}/01b_null_distribution"
os.makedirs(OUT, exist_ok=True)

done = {os.path.basename(f).replace("_result.json", "")
        for f in os.listdir(OUT) if f.endswith("_result.json")} if os.path.isdir(OUT) else set()
print("already complete:", len(done), "target:", len(SEEDS))


In [ ]:
results = []
t_start = time.time()
for seed in SEEDS:
    name = f"{MODEL}__{STRAT}__seed{seed}"
    if name in done:
        print("skip", name)
        continue
    r = engine.run_experiment(ROOT, MODEL, STRAT, seed, OUT)
    results.append(r)
    print(f"    elapsed {(time.time()-t_start)/60:.2f} min")


In [ ]:
import numpy as np
import statistics as st

all_res = engine.collect_results(OUT)
accs = [r["accuracy"] for r in all_res]
ben_f1s = [r["benign_f1"] for r in all_res]
seeds_run = [r["seed"] for r in all_res]

print("=" * 60)
print(f"EFFNET_B0 BLOCK_SHUFFLED NULL DISTRIBUTION (n={len(accs)})")
print("=" * 60)
print(f"Mean Accuracy:    {st.mean(accs):.2f}% +/- {st.stdev(accs) if len(accs) > 1 else 0:.2f}%")
print(f"Median Accuracy:  {float(np.median(accs)):.2f}%")
print(f"Min / Max Acc:    {min(accs):.2f}% / {max(accs):.2f}%")
print(f"Mean Benign F1:   {st.mean(ben_f1s):.2f}% +/- {st.stdev(ben_f1s) if len(ben_f1s) > 1 else 0:.2f}%")
print(f"Central 95% CI:   [{np.percentile(accs, 2.5):.2f}%, {np.percentile(accs, 97.5):.2f}%]")
print("-" * 60)
print(f"Target block_contiguous accuracy: {CONTIGUOUS_MEAN_ACC:.2f}%")

# Calculate empirical percentile (fraction of null <= target)
p_le = float(np.mean(np.array(accs) <= CONTIGUOUS_MEAN_ACC)) * 100.0
print(f"Empirical percentile: {p_le:.1f}th percentile")

in_central_95 = (np.percentile(accs, 2.5) <= CONTIGUOUS_MEAN_ACC <= np.percentile(accs, 97.5))
is_borderline = (90.0 <= p_le <= 97.5) or (2.5 <= p_le <= 10.0)

summary = {
    "model": MODEL,
    "strategy": STRAT,
    "n_seeds": len(accs),
    "seeds": seeds_run,
    "accuracies": [round(float(a), 3) for a in accs],
    "benign_f1s": [round(float(b), 3) for b in ben_f1s],
    "mean_acc": round(float(st.mean(accs)), 3),
    "sd_acc": round(float(st.stdev(accs)), 3) if len(accs) > 1 else 0.0,
    "median_acc": round(float(np.median(accs)), 3),
    "ci95_acc": [round(float(np.percentile(accs, 2.5)), 3), round(float(np.percentile(accs, 97.5)), 3)],
    "contiguous_target_acc": CONTIGUOUS_MEAN_ACC,
    "empirical_percentile": round(p_le, 2),
    "in_central_95": bool(in_central_95),
    "is_borderline": bool(is_borderline)
}

with open(f"{OUT}/null_distribution_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nDECISION RULE OUTCOME:")
if in_central_95:
    print("-> block_contiguous is INSIDE the central 95% of the block_shuffled null distribution.")
    print("-> NO position/ordering effect. Contiguous drop was an unlucky single draw. Option 2 is dead.")
else:
    print("-> block_contiguous is OUTSIDE the central 95% of the block_shuffled null distribution.")
    print("-> Split ordering choice effect is statistically established. Reframe as split ordering choice.")

if is_borderline:
    print("-> HALT CONDITION TRIGGERED: Percentile is borderline [90-97.5 or 2.5-10]. Escalate.")

print("=" * 60)
